# Lab 0: Environment Setup & Backend Choice
**Goal of this notebook:** by the end, you will have (1) a verified working environment, and (2) a working *backend*, either real IBM quantum hardware (Path A), a local simulator (Path B), or ideally both. Every course notebook assumes this one ran successfully.

**Choose your path:**
- **Path A: Real hardware + simulation (recommended).** Free IBM account, API key saved locally, and a first run on a real quantum computer today. Used in the Labs 4, 8, and 12 hardware runs.
- **Path B: Simulation only.** Everything runs locally on your own machine and is ready the moment the packages are installed. This path covers the full course content; section 5 explains what simulation gives you and where it differs from real hardware.

You can start with Path B today and add Path A any time later. Nothing else in the course changes.

**Time estimate:** about 10-20 minutes for Path B. Path A takes about the same hands-on time,
plus a wait in the hardware queue (minutes to hours) that happens in the background.

**Start your confusion log now:** create a file named `confusion_log.md` at the root of
this repo (one file for the whole course). Every time something confuses you, add one line:
the lab, what confused you, and, once it clicks, how. This notebook is where the habit
starts; every later lab assumes the file already exists.

---
# 1. Install and verify the environment

Two ways to install the dependencies; pick the one that matches how you got here:

- **Cloned the repository?** This project's environment is managed by
  [uv](https://docs.astral.sh/uv/): dependencies are declared in `pyproject.toml` and locked
  in `uv.lock`. In a terminal, in the repo root, run `uv sync`, then skip the install cell
  below.
- **Working standalone** (a fresh environment, Google Colab, or qBraid)? Run the install cell
  below; it installs the same set directly into the kernel this notebook is running on.

What each package is for:
- `qiskit`: the core, circuits, gates, `Statevector`
- `qiskit-aer`: high-performance local simulators (Path B lives here)
- `qiskit-ibm-runtime`: the connection to real IBM hardware (Path A lives here)
- `matplotlib`: circuit drawings and the plots later in this notebook
- `pylatexenc`: needed for pretty circuit drawings

In [ ]:
# ═══ 1. INSTALL: run once if you have not installed the packages yet ═══
# %pip installs into the exact Python environment this notebook is running on,
# which is why it is preferred over plain !pip inside notebooks.
# Safe to re-run: already-installed packages are skipped.

%pip install --upgrade 'qiskit[visualization]>=2.5.0' qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc

# If anything was installed just now, restart the kernel once so the fresh packages load,
# then continue below.

Now run the smoke test below. **What you will see:** the imports succeeding, followed by
the installed version of Python and each package. A full pass/fail check, with a fix for
anything missing, lives in the wrap-up readiness check at the end of this notebook
(section 6).

In [ ]:
# ═══ 1. VERIFY: environment smoke test ═══
# Every import this notebook needs, gathered here. Run this cell once per kernel session,
# before any other code cell below; every later cell relies on it having already run.
import sys
from getpass import getpass

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pylatexenc

from IPython.display import display

import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

import qiskit_aer
from qiskit_aer import AerSimulator

import qiskit_ibm_runtime
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeFez

print(f"Python              {sys.version.split()[0]}")
print(f"qiskit              {qiskit.__version__}")
print(f"qiskit-aer          {qiskit_aer.__version__}")
print(f"qiskit-ibm-runtime  {qiskit_ibm_runtime.__version__}")
print(f"matplotlib          {matplotlib.__version__}")
print(f"pylatexenc          {pylatexenc.__version__}")

---
# 2. Path A: connect to real IBM quantum hardware

**On Path B today?** Skip from here to section 4, and once you're there, section 4.3 shows
you how to run this same day-one hardware experiment without an account, using a fake
backend. Sections 2 and 3 will be here whenever you want them.

**On Path A, do this today or later, your choice.** Nothing in Week 0 through Lab 3 needs hardware;
the first required run is in Lab 4. Doing it today has two rewards: any account problems
surface now, while there is slack to fix them, and you get to end your very first session by
running a circuit on a real quantum computer. Skipping it today is equally fine: come back to
this section any time before Lab 4.

### 2.1 Create the account and get your credentials (in the browser)
1. Sign up free at **https://quantum.cloud.ibm.com** (the IBM Quantum Platform).
2. From the **dashboard**, create an **API key**, copy it immediately to a safe place; *it is shown only once*.
3. From the **Instances** page, hover over your **Open** instance's **CRN** and copy it too. **Use the Open instance**: it comes with a monthly allowance of free QPU time, plenty for this course. The CRN identifies *which* account instance your jobs bill against, so this course always sets it explicitly, to make sure jobs land on your free Open instance rather than any other instance your account may have.

### 2.2 Save credentials ONCE, the safe way
The cell below uses `getpass` so your key is **never typed into a notebook cell** (notebooks get committed to git, shared with students, posted in issues... a pasted key *will* eventually leak).
`save_account` writes the credentials to `~/.qiskit/qiskit-ibm.json` on your machine; after this one run, every future notebook connects with zero arguments.

<div style="border-left: 4px solid #f1c40f; background-color: #fff9db; padding: 8px 12px; color: #665200;">
👀 <strong>Where did the prompt go?</strong> Each <code>getpass</code> box below appears at the <strong>top of your notebook editor</strong>, not inline in the cell's output, and is easy to miss the first time.
</div>

In [ ]:
# ═══ 2.2 RUN ONCE: save credentials (Path A only) ═══
# Step 1: ask for the credentials. getpass hides what you type,
# so the secret never appears on screen or in the saved notebook.
print("👀 Two prompts are coming. Look at the top of your notebook editor, not this cell, for each input box.")
token = getpass("Paste your IBM Quantum API key (input hidden): ").strip()
crn = getpass("Paste your Open instance's CRN (input hidden): ").strip()

# Step 2: save them to disk (~/.qiskit/qiskit-ibm.json).
# The CRN is always passed explicitly
# set_as_default=True  -> future notebooks can connect with no arguments.
# overwrite=True       -> re-running this cell replaces old credentials instead of erroring.
QiskitRuntimeService.save_account(
    token=token,
    instance=crn,
    set_as_default=True,
    overwrite=True,
)

# Step 3: remove the secrets from the notebook's memory, so nothing lingers
# in variables that a later cell (or a curious student) could print.
del token
del crn

print("✅ Credentials saved, you never need to run this cell again.")

<div style="border-left: 4px solid #f1c40f; background-color: #fff9db; padding: 8px 12px; color: #665200;">
<strong>Security notes (habits worth keeping for every API key you will ever hold):</strong>
<ul>
<li>The saved file <code>~/.qiskit/qiskit-ibm.json</code> is <strong>plain text</strong>. Fine on a personal machine; on shared machines prefer environment variables (<code>QISKIT_IBM_TOKEN</code>, <code>QISKIT_IBM_INSTANCE</code>) set in your shell profile instead of <code>save_account</code>.</li>
<li>Never hardcode the key in a cell, and clear cell outputs before committing notebooks (<code>jupyter nbconvert --clear-output</code>). Add a <code>.gitignore</code> habit for scratch files.</li>
<li>If a key leaks: delete it from the IBM dashboard and create a new one. Keys are free; leaked compute time is not.</li>
</ul>
</div>

In [ ]:
# ═══ 2.3 VERIFY: connect and list real quantum computers ═══
service = QiskitRuntimeService()  # no arguments, reads the saved credentials
backends = service.backends(operational=True, simulator=False)
print(f"✅ Connected. {len(backends)} real quantum systems visible to your account:")
for b in backends:
    print(f"   {b.name:<20} {b.num_qubits:>4} qubits   pending jobs: {b.status().pending_jobs}")

lb = service.least_busy(operational=True, simulator=False)
print(f"\nShortest queue right now: {lb.name}")

> **About the free (Open) plan:** it includes a limited allowance of QPU execution time per month, jobs wait in a shared queue (minutes to hours), and unused time doesn't roll over. That's plenty for this course, the hardware runs in Labs 4, 8, and 12 are seconds of QPU time each. Check your dashboard for your current allowance and usage. **Habit to build now:** debug on the simulator, and send a job to hardware only when the simulated version already works.

---
# 3. Your first run on a real quantum computer (two acts)

**You are not expected to understand this. Not one line of it.** You are running it for one
reason: to see, with your own eyes, a real quantum computer do something no explanation has
earned yet. Your only job is to *observe* and write down what you notice. Resist the urge to
explain; every explanation you could reach for today is wrong in an interesting way.

Here is the plan, three steps:

1. **Rehearsal.** You build two tiny quantum programs and run them on a perfect simulator on
   your own machine, so you know what *should* happen.
2. **Submission.** You send both programs to a real quantum computer in an IBM lab and get a
   ticket number while they wait in line.
3. **Results.** You collect the answers, compare them to the rehearsal, and write down what
   you saw.

The two programs differ by exactly one thing:

- **Act 1** applies one operation (called H) to a qubit and measures it, 1000 times.
- **Act 2** applies the same operation TWICE, then measures, 1000 times.

### 3.1 The rehearsal, on a perfect simulator

Run the next cell. It builds Act 1 alone, draws the circuit, and runs it 1000 times on a
flawless simulator. **What you will see:** the circuit diagram, then a *counts dictionary*
like `{'0': 512, '1': 488}`, which reads: out of 1000 runs, the outcome was `0` in 512 of
them and `1` in 488. Counts are the native language of quantum computers; you will read
thousands of these over the course, and this is your first.

In [ ]:
# ═══ 3.1 step 1: Act 1 alone, the IDEAL answer from a simulator ═══
act1 = QuantumCircuit(1)
act1.h(0)
act1.measure_all()

print("Act 1 (one H):")
display(act1.draw("mpl"))

simulator = AerSimulator()
result = simulator.run(transpile(act1, simulator), shots=1000).result()
print(f"Act 1, ideal counts: {result.get_counts()}")

**What you just saw:** close to half-and-half, `0` about as often as `1`, like a fair coin
flip. One H turns a qubit that started in a definite state into something that comes out
random when measured.

**Before running Act 2, predict:** if one H looks like a coin flip, what do you think the
SAME operation done TWICE in a row will do? Write your prediction, then run the next cell.

> My prediction: ...

In [ ]:
# ═══ 3.1 step 2: Act 2 alone, the IDEAL answer from a simulator ═══
act2 = QuantumCircuit(1)
act2.h(0)
act2.barrier(0)  # stops the transpiler from quietly cancelling H·H = identity; both gates must actually run
act2.h(0)
act2.measure_all()

print("Act 2 (two Hs):")
display(act2.draw("mpl"))

simulator = AerSimulator()
result = simulator.run(transpile(act2, simulator), shots=1000).result()
print(f"Act 2, ideal counts: {result.get_counts()}")

**Pause here.** Whatever you predicted, sit with what Act 2 actually did for a moment, and
jot your reaction below. No explaining yet; that is Lab 1's job.

> What I noticed: ...

### 3.2 Send both programs to a real machine

Now the same two programs go to an actual quantum computer, a physical device near absolute
zero in an IBM lab. Run the next cell. 

**What you will see, in order:**

- The **name of the machine** chosen for you (the one with the shortest line right now) and
  its qubit count. That name refers to real hardware you could look up on your IBM dashboard.
- A **job id**: your ticket number. Jobs from everyone in the world share these machines, so
  yours waits in a queue: minutes to hours depending on traffic.

**What to do while you wait:** nothing here requires watching.
You can see your job's progress on the Workloads page of your IBM dashboard.

**If the cell fails instead:** the most common cause is missing credentials, fixed by
returning to section 2.2. A long queue is normal and needs no fixing.

In [ ]:
# ═══ 3. step 2: submit both circuits to a real quantum computer ═══
service = QiskitRuntimeService()                                  # uses your saved credentials
backend = service.least_busy(operational=True, simulator=False)   # shortest queue right now
print(f"Sending both circuits to: {backend.name} ({backend.num_qubits} qubits)")

# Real devices only run circuits adapted to their native gates: that is what transpile does
act1_for_device = transpile(act1, backend)
act2_for_device = transpile(act2, backend)

sampler = SamplerV2(mode=backend)
# Tag the job so you can find it later on your IBM dashboard or via service.jobs(job_tags=[...]),
# separate from any other jobs on your account.
sampler.options.environment.job_tags = ["ket-zero", "lab-0"]
job = sampler.run([act1_for_device, act2_for_device], shots=1000)
print(f"Job submitted. Ticket (job id): {job.job_id()}")
print("Your circuits are now in line. Wait for it to finish.")

In [ ]:
# ═══ 3. optional: check on your job any time ═══
# Run this cell whenever you like. When it says DONE, step 3 is ready.
print(f"Job status: {job.status()}")

### 3.3 Collect the results

When the status above says DONE, run the next cell. 

**What you will see:** the same two
counts dictionaries as the rehearsal, but this time every number came from a physical qubit,
followed by a bar chart (a *histogram*) showing both acts side by side: outcomes along the
bottom, how often each occurred as the height of its bar.

**What to watch for, three things:**

1. **Act 1 on hardware vs. Act 1 in rehearsal.** Close? They should be.
2. **Act 2 on hardware vs. Act 2 in rehearsal.** Mostly the same story, with one difference:
   look for a small bar that the rehearsal said should not exist at all.
3. **That small bar.** The perfect simulator gave a clean answer; the real machine almost,
   but not quite, agrees. Remember it. It is the most important little bar in this course.

Save the histogram (right-click it, or screenshot): you will want to look back at it during
Lab 1.

In [ ]:
# ═══ 3. step 3: collect your results (run this after the job finishes) ═══
result = job.result()
counts_act1 = result[0].data.meas.get_counts()
counts_act2 = result[1].data.meas.get_counts()

print("Act 1 (one H), real hardware: ", counts_act1)
print("Act 2 (two Hs), real hardware:", counts_act2)

# Save this figure: you will look back at it in Lab 1.
plot_histogram([counts_act1, counts_act2], legend=["Act 1: one H", "Act 2: two Hs"])

## 📝 Day-one observation log (observations only, no explanations)

- Act 1 gave me roughly: ...
- My prediction for Act 2 was: ...
- Act 2 actually gave me: ...
- Compared to the rehearsal, the real machine differed by: ...
- The strangest part, in one sentence: ...

Three things you just witnessed, stated without explanation. One H looks like a fair coin
flip. The same "coin flip" done twice gives (almost) always 0: whatever H does, doing it
twice undoes it, which no coin can do. And the real machine disagrees slightly with the
perfect simulator: that small extra bar in Act 2 is your first sighting of *noise*.

Lab 1 pays the debt for Acts 1 and 2. The noise gets a name in Unit III, and Unit IV is
about defeating it. Welcome to the course.

*Path B note: the same three steps work without an account by swapping the real backend for
`FakeFez` (section 4.3 shows how). You get the noise, though admittedly some of the
goosebumps require the real machine.*

---
# 4. Path B: local simulation, entirely on your own machine

Three local tools cover everything, and **you will use them constantly even if you have
hardware access**: simulators are where all development happens.

### 4.1 Exact statevector: the microscope

Run the next cell. It rebuilds Act 2 from section 3 (two H gates), and instead of measuring,
asks the simulator directly for the full quantum state.

**What you will see:** the two raw amplitudes, one per outcome (`0` and `1`). Since two H
gates undo each other, expect amplitude 1 on `0` and 0 on `1`, then the measurement
probabilities computed from them. No real quantum computer will ever show you this directly,
only a simulator can, which is exactly why it's called "the microscope."

In [ ]:
# ═══ 4.1 Exact statevector: the microscope ═══
# Perfect, noiseless, and you can inspect the full quantum state (impossible on real hardware!).
# Same circuit as Act 2 in section 3: two H gates, with a barrier so transpiling for the
# noisy simulator in 4.3 can't quietly cancel them out as H·H = identity.
qc = QuantumCircuit(1)
qc.h(0)
qc.barrier(0)
qc.h(0)

sv = Statevector.from_instruction(qc)
print("Full state (cheating, nature never shows you this):", np.round(sv.data, 4))
print("Measurement probabilities:", sv.probabilities_dict())

### 4.2 AerSimulator: the hardware stand-in

Run the next cell. It takes the same circuit, adds a measurement, and runs it 1000 times
through `AerSimulator`, the way a real device would: shots in, counts out, no peeking at the
state in between.

**What you will see:** a counts dictionary close to `{'0': 1000}`, the same near-certain
result you already saw from Act 2 in section 3, now reproduced entirely on your own machine.
This is your default backend for all of Path B, fast, noiseless, and the one you'll use
constantly even once you also have hardware access.

In [ ]:
# ═══ 4.2 AerSimulator: the hardware stand-in ═══
# Runs shots like a real device: you get counts, not the state. This is your default backend for Path B.
qc_meas = qc.copy()
qc_meas.measure_all()

sim = AerSimulator()
result = sim.run(transpile(qc_meas, sim), shots=1000).result()
print("Counts from 1000 shots:", result.get_counts())
# Expect close to {'0': 1000}, the same near-certain result as Act 2 in section 3.

### 4.3 Noisy simulation: a fake quantum computer

Run the next cell. It runs the same circuit again, but this time on a *fake backend*: a
snapshot of a real IBM device's calibration data (a Heron r2 system, `ibm_fez`), replayed
locally. This cell uses 20,000 shots instead of 1,000: the gate error on a well-calibrated
qubit is small enough that 1,000 shots can come back perfectly clean by chance, and locally
the extra shots cost nothing but a moment of compute.

**What you will see:** almost certainly, a small number of `1` outcomes mixed in with the
`0`s, the same "small bar that shouldn't exist" you saw from Act 2's real-hardware run in
section 3, this time on your own machine. That's simulated noise. (On the rare run where it
comes back perfectly clean again, that's still a valid result, just rerun the cell for a
fresh sample.) This is how Path B users experience the "hardware" portions of Labs 4, 8,
and 12 without an IBM account.

In [ ]:
# ═══ 4.3 Noisy simulation: a fake quantum computer ═══
# Snapshots of real IBM devices (calibration data included) let you simulate realistic noise offline.
# This is how Path B users do the 'hardware' portions of Labs 4, 8, and 12.
# More shots than section 3's real hardware run: gate error here is small, so a low shot
# count can come back perfectly clean by chance. Locally, extra shots are free.
fake = FakeFez()   # a snapshot of a real Heron r2 device (ibm_fez)
result = fake.run(transpile(qc_meas, fake), shots=20000).result()
print("Counts on the FAKE noisy device:", result.get_counts())
# Now you may see some '1' counts, noise! Compare with the perfect AerSimulator counts above.

---
# 5. What simulation gives you, and where it differs from real hardware

### Limitation 1: exponential memory, the wall is real
An n-qubit state is 2ⁿ complex amplitudes (16 bytes each). Nothing negotiates with that exponent, run the cell below and find where YOUR machine dies. (For this course it's irrelevant: nothing here exceeds ~10 qubits. It becomes the whole story if you later simulate serious algorithms, which is, of course, exactly why quantum computers are interesting.)

In [ ]:
# ═══ 5. BUILD: the exponential wall, visualized ═══
# The reasoning, step by step:
#   - a state of n qubits is a vector with 2^n complex amplitudes
#   - each complex amplitude is two 64-bit floats = 16 bytes
#   - so total memory = 2^n * 16 bytes
BYTES_PER_AMPLITUDE = 16
BYTES_PER_GIB = 2**30          # 1 GiB = 2^30 bytes

qubit_counts = []
memory_in_gib = []
for n in range(1, 51):
    number_of_amplitudes = 2**n
    total_bytes = number_of_amplitudes * BYTES_PER_AMPLITUDE
    total_gib = total_bytes / BYTES_PER_GIB
    qubit_counts.append(n)
    memory_in_gib.append(total_gib)

# Plot it (log scale on y, because the growth is exponential)
plt.figure(figsize=(8, 4.5))
plt.semilogy(qubit_counts, memory_in_gib, marker='.')

# Draw reference lines for real machines, one at a time
plt.axhline(16, linestyle='--', linewidth=0.8)          # a typical laptop: 16 GiB
plt.text(1, 16 * 1.5, "laptop (16 GiB)", fontsize=8)

plt.axhline(1024, linestyle='--', linewidth=0.8)        # a big server: 1 TiB = 1024 GiB
plt.text(1, 1024 * 1.5, "big server (1 TiB)", fontsize=8)

plt.axhline(1e9, linestyle='--', linewidth=0.8)         # ~all the RAM on Earth (order of magnitude)
plt.text(1, 1e9 * 1.5, "roughly all RAM on Earth", fontsize=8)

plt.xlabel("number of qubits")
plt.ylabel("memory needed for the statevector (GiB, log scale)")
plt.title("Why simulators hit a wall around 30-45 qubits")
plt.tight_layout()
plt.show()

# Print a few landmark values to make it concrete
for n in [20, 30, 40, 50]:
    total_gib = (2**n * BYTES_PER_AMPLITUDE) / BYTES_PER_GIB
    print(f"{n} qubits -> {total_gib:,.1f} GiB just to STORE the state (before doing any computation)")

### Limitation 2: simulated noise is a model, not the messy truth
Fake backends replay a *calibration snapshot*. Real devices drift hour to hour, have readout errors, crosstalk, and occasional surprises no model captures. If you only ever simulate, quantum computing feels cleaner than it is, one real hardware run teaches a humility that no simulator can.

### Limitation 3: the statevector simulator lets you cheat
`Statevector` shows you all amplitudes at once. Real quantum mechanics never does, you get one measurement outcome per shot, full stop. This "god view" is a *fantastic learning tool* (the course notebooks exploit it constantly) but be conscious you're cheating: any reasoning that requires seeing the state is reasoning a real quantum computer can't do. Good self-check when designing algorithms: *could I still do this with counts only?*

### Limitation 4: real devices have queues, connectivity limits, and native gate sets
Real devices have limited qubit connectivity (your CNOT between distant qubits silently becomes a chain of SWAPs), restricted native gate sets, and shared queues. Simulation hides all of this. Labs 4/8/12 hardware runs exist precisely to surface it.

### Bottom line
| | Path B (simulation only) | Path A (+ hardware) |
|---|---|---|
| Course concepts & all 16 labs | ✅ fully | ✅ fully |
| Noise experience | 🟡 modeled (fake backends) | ✅ the real thing |
| Cost / setup | ✅ zero | ✅ free tier, small setup |
| "I ran this on an actual quantum computer" | ❌ | ✅ (priceless for motivation & teaching) |

**Recommendation:** develop everything on simulators regardless of path; if you can, add Path A for the three hardware touchpoints. For classrooms without accounts, Path B + fake backends is a legitimate, complete course experience.

---
# 6. 🎓 Wrap-up: final readiness check

Run the cell below. It detects your setup and tells you exactly where you stand.

In [ ]:
# ═══ Readiness check ═══
# Check 1: is the Qiskit core installed? (required for everything)
try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
    core_works = True
except ImportError:
    core_works = False

if core_works:
    print("✅ Qiskit core          -> required for everything")
else:
    print("❌ Qiskit core missing  -> run: pip install qiskit")

# Check 2: is the local simulator installed? (this is Path B)
try:
    from qiskit_aer import AerSimulator
    simulation_works = True
except ImportError:
    simulation_works = False

if simulation_works:
    print("✅ Local simulation     -> Path B ready")
else:
    print("❌ Local simulation missing -> run: pip install qiskit-aer")

# Check 3: can we reach real IBM hardware? (this is Path A)
# Note: this one can fail for several reasons (package not installed,
# no saved credentials, no internet), so we catch ANY exception,
# and failing is perfectly fine - Path B covers the whole course.
try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()   # reads saved credentials from disk
    service.backends()                 # actually talks to IBM to prove it works
    hardware_works = True
except Exception:
    hardware_works = False

if hardware_works:
    print("✅ IBM hardware access  -> Path A ready")
else:
    print("⚪ IBM hardware access  -> Path A not configured (fine: Path B covers the course)")

if core_works and simulation_works:
    print()
    print("🚀 You are ready. Next stop: warmup0a_sets_functions_bits.ipynb")


# Additional information

**Created by:** Jesús Hernández Tapia

**Version:** 1.0.0